# KLA/i4C Restoration — Visualization Notebook

This notebook is for **visualization only**. All training/evaluation logic lives in the `.py` modules (`train.py`, `evaluate.py`, `models/`, `losses/`, `metrics/`) so the pipeline stays reproducible outside Jupyter. Run this after training to inspect: training curves, sample restorations, and normalization behavior.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from utils.image_utils import read_image_grayscale, normalize_degraded, normalize_gt
from evaluate import load_model_and_config, resolve_device

## 1. Training curves (from CSV log)

In [ ]:
log_path = Path('../runs/train_log.csv')
if log_path.exists():
    df = pd.read_csv(log_path)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(df['epoch'], df['train_loss'], label='train')
    axes[0].plot(df['epoch'], df['val_loss'], label='val')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(df['epoch'], df['val_psnr'])
    axes[1].set_title('Validation PSNR (dB)')
    axes[2].plot(df['epoch'], df['val_ssim'])
    axes[2].set_title('Validation SSIM')
    plt.tight_layout(); plt.show()
else:
    print(f'No log found at {log_path} yet — run train.py first.')

## 2. Inspect the intensity-normalization behavior on a sample pair
Confirms out-of-range degraded pixels are preserved (not clamped) before the model sees them — see `utils/image_utils.py`.

In [ ]:
degraded_dir = Path('../dataset/val/degraded')
gt_dir = Path('../dataset/val/ground_truth')
sample = sorted(degraded_dir.glob('*'))[0] if degraded_dir.exists() else None
if sample:
    degraded = read_image_grayscale(sample)
    norm, stats = normalize_degraded(degraded)
    print(f'Normalized range: [{norm.min():.3f}, {norm.max():.3f}] (values outside [0,1] are intentional)')
    plt.hist(norm.ravel(), bins=100)
    plt.title('Normalized degraded-pixel histogram')
    plt.show()
else:
    print('No validation data found yet.')

## 3. Qualitative restoration preview
For a full comparison grid across many images, prefer `scripts/generate_comparisons.py` (saves to `outputs/comparisons/`). This cell is a quick single-image check.

In [ ]:
weights_path = Path('../weights/best_model.pth')
if weights_path.exists() and sample:
    device = resolve_device('auto')
    model, cfg, ckpt = load_model_and_config(weights_path, '../config/config.yaml', device)
    state = ckpt.get('ema_state_dict') or ckpt.get('model_state_dict', ckpt)
    model.load_state_dict(state)
    model.to(device).eval()

    x = torch.from_numpy(norm).unsqueeze(0).unsqueeze(0).float().to(device)
    with torch.inference_mode():
        pred = model(x).squeeze().cpu().numpy()

    gt_path = gt_dir / sample.name
    fig, axes = plt.subplots(1, 3 if gt_path.exists() else 2, figsize=(12, 4))
    axes[0].imshow(np.clip(norm, 0, 1), cmap='gray'); axes[0].set_title('Degraded'); axes[0].axis('off')
    axes[1].imshow(np.clip(pred, 0, 1), cmap='gray'); axes[1].set_title('Restored'); axes[1].axis('off')
    if gt_path.exists():
        gt_norm, _ = normalize_gt(read_image_grayscale(gt_path))
        axes[2].imshow(gt_norm, cmap='gray'); axes[2].set_title('Ground Truth'); axes[2].axis('off')
    plt.tight_layout(); plt.show()
else:
    print('No trained weights found yet at ../weights/best_model.pth — train the model first.')